# Bloc 5 — Le Nettoyage de Données : L'Art Invisible

Message clé : un data scientist passe 60-80% de son temps à nettoyer les données.
Objectifs: diagnostiquer les NaN, stratégies d'imputation, détecter les doublons, identifier les outliers, remplacer et mapper des valeurs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Chargement du dataset À NETTOYER 
df_dirty = pd.read_csv('../01_data/exercises/thelook_sales_dirty.csv')
print(f"Dataset chargé : {df_dirty.shape[0]} lignes × {df_dirty.shape[1]} colonnes")

Dataset chargé : 51000 lignes × 20 colonnes


## 1. Diagnostiquer les valeurs manquantes (NaN)
La première étape du nettoyage est toujours le diagnostic. Avant de corriger, il faut comprendre.

### Détecter les NaN

In [72]:
# Quelles colonnes ont des NaN ?
df_dirty.isna().sum()


id                    0
order_id              0
user_id               0
product_id            0
sale_price         4039
status                0
created_at            0
shipped_at         8038
delivered_at      14345
returned_at       47403
product_name          0
category           6720
brand                 0
department            0
cost                  0
gender                0
age                2502
country               0
city                  0
traffic_source        0
dtype: int64

### Quantifier en pourcentage

In [73]:
# Le pourcentage est plus parlant que le nombre absolu
dirty_percent = (df_dirty.isna().mean() * 100).round(1)
dirty_percent

id                 0.0
order_id           0.0
user_id            0.0
product_id         0.0
sale_price         7.9
status             0.0
created_at         0.0
shipped_at        15.8
delivered_at      28.1
returned_at       92.9
product_name       0.0
category          13.2
brand              0.0
department         0.0
cost               0.0
gender             0.0
age                4.9
country            0.0
city               0.0
traffic_source     0.0
dtype: float64

### Visualiser les NaN

In [ ]:
# Que constatez-vous sur ce graphique ?
na_values_mean = df_dirty.isna().mean().sort_values()

# creation d'un bar plot horizontal pour mieux visualiser les valeurs manquantes
barplot = na_values_mean.plot.barh()
plt.title('Pourcentage de valeurs manquantes par colonne')
plt.xlabel('% NaN')
plt.show()

## 2. Stratégies de traitement des NaN : Supprimer ou Remplacer ?

> 🎯 **Règle d'or du Data Analyst :** Le choix entre **supprimer (`.dropna`)** et **remplacer (`.fillna`)** n'est jamais un choix technique ou mathématique au hasard, c'est une **décision 100% MÉTIER**.  
> Une mauvaise imputation fausse les métriques financières ; une suppression aveugle élimine des clients réels.

---

### Cas Pratique : Le Dilemme du Bilan Financier chez *TheLook*

La Direction Financière vous mandate pour calculer :
1. Le **Chiffre d'Affaires global** généré.
2. Le **Panier moyen** et le **profil d'âge** des acheteurs.

En ouvrant `thelook_sales_dirty.csv`, vous constatez des valeurs manquantes sur 4 colonnes clés :
- `returned_at` (92.9% de NaN)
- `sale_price` (7.9% de NaN — soit plus de 4 000 lignes)
- `age` (4.9% de NaN — soit 2 500 clients)
- `category` (13.2% de NaN)

---

### 🧠 Analyse Décisionnelle Métier : Que faire pour chaque colonne ?

| Colonne | % NaN | Question Métier | Décision & Justification |
| :--- | :---: | :--- | :--- |
| **`returned_at`** | **92.9%** | Le taux de NaN est massif. Faut-il supprimer la colonne ou les lignes ? | ❌ **Ne surtout pas supprimer !**<br>Un `NaN` ici n'est pas une donnée manquante : **c'est une excellente nouvelle métier** (la commande n'a pas été retournée). Supprimer les `NaN` supprimerait 93% des ventes saines !<br>➡️ *Action : Créer un booléen `is_returned` avec `.notna()`.* |
| **`age`** | **4.9%** | Faut-il supprimer les commandes dont le client n'a pas renseigné son âge ? | ❌ **Ne pas supprimer la ligne !**<br>Une commande reste valide même si l'âge est inconnu. Supprimer 2 500 lignes ferait disparaître artificiellement des milliers d'euros de ventes.<br>➡️ *Action : Remplacer par la médiane de l'âge (`median()`).* |
| **`category`** | **13.2%** | Faut-il jeter les produits sans catégorie ? | ❌ **Non.** On perdrait la visibilité sur les volumes de vente.<br>➡️ *Action : Imputer par une catégorie textuelle explicite : `'Non renseigné'`.* |
| **`sale_price`** | **7.9%** | Peut-on inventer un prix de vente avec la moyenne ou la médiane ? | ⚠️ **Dilemme critique :**<br>• Si le but est d'auditer le **CA comptable réel** ➡️ On **supprime** les lignes sans prix (`.dropna(subset=['sale_price'])`) car on ne facture pas des médianes théoriques.<br>• Si le but est d'estimer un **volume potentiel** ➡️ On **remplace** par la médiane du prix *de la même catégorie*. |

---

### 🛠️ Mise en Pratique : Le Script de Nettoyage

Testons ces différentes stratégies sur une copie de travail :


### Supprimer les lignes : .dropna()

In [ ]:
df_test = df_dirty.copy()
df_test = df_test.dropna(subset=['sale_price'])

# On perd des lignes ! Est-ce acceptable ?
print(f"Lignes après dropna: {df_test.shape[0]}")

In [ ]:
# Il peut être interressant de savoir le nombre de lignes supprimé en %
pct_supprime = (len(df_dirty) - len(df_test)) / len(df_dirty) * 100

print(f"Lignes supprimées : {pct_supprime:.2f}%")

Lignes supprimées : 7.92%


💡 Astuce Python : Le modificateur :.1% (ou :.2%) dans une f-string multiplie automatiquement le résultat par 100, arrondit à 1 décimale et ajoute le symbole %.

### Remplir par une valeur fixe : .fillna()

In [76]:
# Remplir par 0 est rarement une bonne idée pour des prix...
df_test['sale_price'] = df_test['sale_price'].fillna(0)
display(df_test[['id', 'sale_price']].head())

# Pourquoi selon vous ?

,id,sale_price
0,1,37.47
1,2,41.01
2,3,15.67
3,4,75.15
4,5,18.20


### Remplir par la médiane ( Data imputation)

In [77]:
# On utilise la mediane plutot que la moyenne 
# Expliquez pourquoi

mediane = df_dirty['sale_price'].median()
df_dirty['sale_price'] = df_dirty['sale_price'].fillna(mediane)
print(f"Médiane utilisée pour l'imputation : {mediane}")

Médiane utilisée pour l'imputation : 47.03


### Remplir par la valeur précédente/suivante

In [84]:
# Exemple typique : Relevé quotidien (ex: cours de bourse ou capteur météo avec jours manquants)
dates = pd.date_range(start='2024-01-01', periods=6, freq='D')
mesures = pd.Series([100.0, np.nan, np.nan, 130.0, np.nan, 150.0], index=dates)

# Comparaison côte à côte de ffill() et bfill()
comparatif = pd.DataFrame({
    'Brut (avec NaN)': mesures,
    'ffill (Reporter valeur précédente)': mesures.ffill(),
})

comparatif


,Brut (avec NaN),ffill (Reporter valeur précédente)
2024-01-01,100.0,100.0
2024-01-02,NaN,100.0
2024-01-03,NaN,100.0
2024-01-04,130.0,130.0
2024-01-05,NaN,130.0
2024-01-06,150.0,150.0


Quelle stratégie choisiriez-vous pour la colonne `age` ? Et pour `sale_price` ? Justifiez.

## 3. Détecter et traiter les doublons

### Combien de doublons exacts ?

In [ ]:
nb = df_dirty.duplicated().sum()
print(f"{nb} lignes dupliquées exactement")

### Supprimer les doublons

In [85]:
df_clean = df_dirty.drop_duplicates()
print(f"Avant: {df_dirty.shape[0]}, Après: {df_clean.shape[0]}")

Avant: 51000, Après: 50515


### Doublons partiels (sur certaines colonnes)

In [86]:
# n'hesitez à utliser la methode pour obtenir le nom des colonnes
df_test."method"

SyntaxError: invalid syntax (913927280.py, line 2)

In [87]:
# Combien de doublons si on ne regarde que l'ID des commandes et des produits ?
valeur1 = "order_id"
valeur2 = "product_id"

nb_partiels = df_dirty.duplicated(subset=[valeur1,valeur2]).sum()    #remplace par les valeurs adaptés 

print(f"{nb_partiels} doublons sur {valeur1}, {valeur2})")

1036 doublons sur order_id, product_id)


## 4. Détecter les outliers (valeurs aberrantes)

### Repérer avec .describe()

In [88]:
# Regardez le min et le max. Quelque chose vous choque ?
df_dirty['sale_price'].describe()

count    51000.000000
mean        88.751362
std        521.451475
min        -99.640000
25%         27.900000
50%         47.030000
75%         75.530000
max      14988.280000
Name: sale_price, dtype: float64

### La Méthode IQR (Écart Interquartile) : Détecter les Outliers

L'**IQR** (*Interquartile Range*) est une méthode statistique pour identifier les **valeurs aberrantes (outliers)** sans être trompé par les valeurs extrêmes.

---

#### 1. Le Principe
On divise les données triées en 4 quarts (quartiles) :
* **$Q_1$ (25% des données) :** Le premier quartile.
* **$Q_3$ (75% des données) :** Le troisième quartile.
* **$\text{IQR} = Q_3 - Q_1$ :** L'intervalle qui contient les **50% centraux** des données.

_Ceci est l'un des indicateurs utilisés pour approcher la centrailité d'un jeu de données_

---

#### 2. La Règle des 1,5 × IQR (Règle de Tukey)
Toute valeur en dehors de la « zone normale » est considérée comme un **outlier** :

$$\text{Borne inférieure} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Borne supérieure} = Q_3 + 1.5 \times \text{IQR}$$

```text
               Zone Normale (99.3% si loi normale)
         [---------------------------------]
---x-----|---------|=========|-------------|------x---
Outlier  Borne    Q1     Médiane   Q3    Borne  Outlier
         Basse                           Haute


In [ ]:
Q1 = df_dirty['sale_price'].quantile(0.25)
Q3 = df_dirty['sale_price'].quantile(0.75)
IQR = Q3 - Q1

# Bornes
borne_basse = Q1 - 1.5 * IQR
borne_haute = Q3 + 1.5 * IQR

# Détection des outliers
masque_outliers = (df_dirty['sale_price'] < borne_basse) | (df_dirty['sale_price'] > borne_haute)
print(f"Nombre d'outliers détectés : {masque_outliers.sum()}")


## 5. Remplacer et mapper des valeurs

### Corriger des valeurs incohérentes avec .replace()

In [ ]:
# Observez les variantes : US, USA, united states...
df_dirty['country'].value_counts().head(20)

### Créer un dictionnaire de mapping

In [90]:
# On va dábord nettoyer les espaces avant et après le texte
df_dirty['country'] = df_dirty['country'].str.strip()

df_dirty['country'].value_counts().head(20)

country
United States     16319
China              3443
Brazil             3058
United Kingdom     2639
France             2349
Spain              2128
Australia          2099
Germany            1975
Austria            1794
Japan              1778
Mexico             1637
Belgium            1582
South Korea        1566
Colombia           1535
Poland             1442
USA                1055
UK                  558
brazil              443
BRAZIL              434
Brasil              433
Name: count, dtype: int64

In [89]:
# Un dictionnaire permet de standardiser proprement
mapping = {
    'U.S.A.': 'United States', 
    'US': 'United States', 
    'united states': 'United States',
    '  United States': 'United States'}

df_dirty['country'] = df_dirty['country'].replace(mapping)

df_dirty['country'].value_counts().head(20)

country
United States        15317
China                 3443
Brazil                3058
United Kingdom        2639
Spain                 2128
France                2112
Australia             2099
Austria               1794
Germany               1791
Japan                 1778
Mexico                1637
Belgium               1582
South Korea           1566
Colombia              1535
Poland                1442
USA                   1055
  United States       1002
brazil                 443
BRAZIL                 434
Brasil                 433
Name: count, dtype: int64

### Recoder avec .map()

In [ ]:
# map() est parfait pour transformer une catégorie en une autre
gender_map = {'M': 'Male', 'F': 'Female'}

df_dirty['gender_full'] = df_dirty['gender'].map(gender_map)
df_dirty[['gender', 'gender_full']].head()

### ⚖️ Différence clé : `.replace()` vs `.map()`

La différence majeure réside dans le comportement face aux **valeurs qui NE sont PAS dans le dictionnaire** :

| Méthode | Ce qu'elle fait aux valeurs absentes du dictionnaire | Utilisation idéale |
| :--- | :--- | :--- |
| **`.replace()`** | **Conserve les valeurs d'origine intactes** | **Nettoyage ponctuel** : corriger quelques fautes de frappe sans toucher au reste des données. |
| **`.map()`** | **Les transforme toutes en `NaN` !** | **Recodage exhaustif** : quand toutes les catégories possibles sont connues à 100%. |

#### 🧪 Exemple illustratif :

In [ ]:
# Imaginons la série suivante :
s = pd.Series(['US', 'France', 'spain'])
mapping = {'US': 'United States', 'spain': 'Spain'}

In [ ]:
s.replace(mapping)
# Résultat :
# 0    United States
# 1           France   <-- 'France' est CONSERVÉ intact !
# 2            Spain


In [ ]:
s.map(mapping)
# Résultat :
# 0    United States
# 1              NaN   <-- ATTENTION : 'France' est devenu NaN car absent du dictionnaire !
# 2            Spain


### ⚖️ Différence clé : `.replace()` vs `.map()`

La différence majeure réside dans le comportement face aux **valeurs qui NE sont PAS dans le dictionnaire** :

| Méthode | Ce qu'elle fait aux valeurs absentes du dictionnaire | Utilisation idéale |
| :--- | :--- | :--- |
| **`.replace()`** | **Conserve les valeurs d'origine intactes** | **Nettoyage ponctuel** : corriger quelques fautes de frappe sans toucher au reste des données. |
| **`.map()`** | **Les transforme toutes en `NaN` !** | **Recodage exhaustif** : quand toutes les catégories possibles sont connues à 100%. |

#### 🧪 Exemple illustratif :

In [ ]:
# Imaginons la série suivante :

s = pd.Series(['US', 'France', 'spain'])
mapping = {'US': 'United States', 'spain': 'Spain'}

In [92]:
s.replace(mapping)
# Résultat :
# 0    United States
# 1           France   <-- 'France' est CONSERVÉ intact !
# 2            Spain


0    United States
1           France
2            Spain
dtype: str

In [93]:
s.map(mapping)
# Résultat :
# 0    United States
# 1              NaN   <-- ATTENTION : 'France' est devenu NaN car absent du dictionnaire !
# 2            Spain


0    United States
1              NaN
2            Spain
dtype: str

### Nettoyer la casse avec .str

In [94]:
# La casse mélangée crée des faux doublons dans les catégories
df_dirty['category'].str.lower().value_counts()

category
jumpsuits & rompers              2893
pants                            2667
sweaters                         2403
outerwear & coats                2368
blazers & jackets                2363
shorts                           2357
fashion hoodies & sweatshirts    2341
leggings                         2338
dresses                          2325
accessories                      2324
jeans                            2292
suits & sport coats              2289
swim                             2276
active                           2247
skirts                           2212
sleep & lounge                   2200
socks & hosiery                  2135
tops & tees                      2128
underwear                        2122
Name: count, dtype: int64

---
# Exercices pratiques

## Exercice 5.1 — L'Audit Qualité
Produisez un rapport qualité complet du dataset `thelook_sales_dirty.csv` :
1. % de NaN par colonne
2. Nombre de doublons exacts + doublons sur (order_id, product_id)
3. Colonnes avec des types suspects (cost est-il numérique ?)
4. Valeurs aberrantes dans sale_price (négatifs ? valeurs > 5000 ?)
5. Combien de variantes pour le pays ? (il devrait y en avoir ~15)

In [ ]:
# 1. Pourcentage de NaN
# Votre code ici

In [ ]:
# 2. Doublons
# Votre code ici

In [ ]:
# 3. Types suspects
# Votre code ici

In [ ]:
# 4. Outliers dans sale_price
# Votre code ici

## Exercice 5.2 — Le Chirurgien data
Appliquez un pipeline de nettoyage complet. Travaillez sur une COPIE (`df = df_dirty.copy()`).
1. Remplacez les NaN de sale_price par la médiane globale
2. Supprimez les doublons exacts
3. Convertissez cost de texte ("$12.50") en float — Astuce : `.str.replace('$', '', regex=False).astype(float)`
4. Plafonnez sale_price au 99e percentile avec .clip()
5. Supprimez les lignes avec sale_price négatif

In [ ]:
# (a) Imputation des NaN de sale_price
df = df_dirty.copy()
# Votre code ici

In [ ]:
# (b) Suppression des doublons
# Votre code ici

In [ ]:
# (c) Conversion de cost
# Votre code ici

In [ ]:
# (d) Plafonnement des outliers
# Votre code ici

In [ ]:
# (e) Suppression des prix négatifs + vérification finale
# Votre code ici
# print(f"Shape final : {df.shape}")

## Exercice 5.3 — L'Harmonisateur de valeurs 
1. Affichez toutes les variantes de pays avec `.value_counts()`
2. Créez un dictionnaire de mapping pour standardiser (US, USA, united states → 'United States', etc.)
3. Appliquez avec `.replace()` et vérifiez le résultat

In [ ]:
# 1. Explorez les variantes
# Votre code ici

In [ ]:
# 2 et 3. Mapping + vérification
# Votre code ici